# AI Tutor Fine-Tuning -- Programming (Python / DSA / SQL)
**Purple Merit Technologies -- AI/ML Engineer Assessment**

Started: 27 March 2026

---

Decided to go with **Programming** as the tutoring category since its what I know best and its the easiest to evaluate objectively (you can just run the code).

Choosing **Gemma-2-2b-it** as base -- fits in Colab T4's 16GB VRAM with 4-bit quantization, and the `-it` (instruction-tuned) checkpoint gives a much better starting point than a raw pretrain.

Plan:
1. Setup
2. Build dataset (mix of synthetic + CodeAlpaca filtered)
3. QLoRA fine-tune
4. Inference demo
5. Evaluation (base vs fine-tuned)
6. Export adapter

## 1. Environment Setup

In [ ]:
!pip uninstall -y peft transformers trl accelerate bitsandbytes 2>/dev/null
!pip cache purge 2>/dev/null
!pip install -q -U transformers==4.44.0 peft==0.12.0 trl bitsandbytes accelerate==0.31.0 datasets scikit-learn
print("Install complete")

Found existing installation: peft 0.18.1
Uninstalling peft-0.18.1:
  Successfully uninstalled peft-0.18.1
Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
Found existing installation: accelerate 1.13.0
Uninstalling accelerate-1.13.0:
  Successfully uninstalled accelerate-1.13.0
Files removed: 0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 66.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.4/296.4 kB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 309.4/309.4 kB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.6/316.6 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 103.4 MB/s 

In [ ]:
import os, json, re, random, warnings, textwrap, subprocess, sys, tempfile, gc
import torch
import numpy as np
from datasets import Dataset, load_dataset
from pathlib import Path
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')
random.seed(42); np.random.seed(42)

assert torch.cuda.is_available(), "Go to Runtime > Change runtime type > T4 GPU"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
DEVICE = 'cuda'

GPU: Tesla T4
VRAM: 15.6 GB


## 2. Configuration

In [ ]:
CFG = {
    'base_model'        : 'google/gemma-2-2b-it',
    'output_dir'        : './prog_tutor_qlora',
    # LoRA
    'lora_r'            : 16,
    'lora_alpha'        : 32,
    'lora_dropout'      : 0.05,
    'target_modules'    : ['q_proj','k_proj','v_proj','o_proj',
                           'gate_proj','up_proj','down_proj'],
    # Training
    'num_train_epochs'  : 5,
    'per_device_batch'  : 1,
    'grad_accum_steps'  : 8,
    'learning_rate'     : 5e-5,
    'lr_scheduler'      : 'cosine',
    'warmup_ratio'      : 0.1,
    'max_seq_length'    : 1024,
    'weight_decay'      : 0.01,
    'max_grad_norm'     : 1.0,
    'bf16'              : False,   # T4 doesn't support bf16
    'logging_steps'     : 10,
    'save_steps'        : 100,
    'eval_steps'        : 100,
    # Inference
    'test_size'         : 0.1,
    'max_new_tokens'    : 512,
    'temperature'       : 0.7,
    'top_p'             : 0.9,
}

os.makedirs(CFG['output_dir'], exist_ok=True)
os.makedirs('./data', exist_ok=True)
print("Config ready")

Config ready


## 3. Dataset

**Tutoring format:** `Goal → Concept → Step-by-step → Worked Example → Checkpoint`

Mix of hand-authored synthetic examples + filtered CodeAlpaca-20k (Apache 2.0).

In [ ]:
SYSTEM_PROMPT = (
    "You are a programming tutor specializing in Python, "
    "Data Structures & Algorithms, and SQL.\n\n"
    "REQUIRED FORMAT — you MUST use this exact structure for every answer:\n"
    "**Goal** — one sentence: what the student will learn\n"
    "**Concept** — intuitive explanation, use analogies\n"
    "**Step-by-step** — numbered steps breaking down the approach\n"
    "**Worked Example** — clean working code in a ```python block\n"
    "**Checkpoint** — end with one question to verify understanding\n\n"
    "RULES:\n"
    "- Always use the bold section headers exactly as shown above\n"
    "- Always include a ```python code block in Worked Example\n"
    "- Always end with a Checkpoint question\n"
    "- Guide students to think; do not just give the answer\n"
    "- Stay within Python/DSA/SQL — redirect out-of-scope questions\n"
    "- Out-of-scope: respond with one line explaining you only cover programming"
)

### 3a. Synthetic Examples

30 hand-authored high-quality examples covering core DSA, Python, SQL, debugging, and out-of-scope redirects. Quality > quantity.

In [ ]:
# Load synthetic examples from JSON
import json

SYNTH_PATH = '/content/synth_data.json'  # adjust if needed

with open(SYNTH_PATH, 'r', encoding='utf-8') as f:
    SYNTHETIC_EXAMPLES = json.load(f)

print(f'Loaded {len(SYNTHETIC_EXAMPLES)} synthetic examples from {SYNTH_PATH}')
# Quick sanity check
print(f'First example instruction: {SYNTHETIC_EXAMPLES[0]["instruction"][:60]}...')
assert all('instruction' in ex and 'response' in ex for ex in SYNTHETIC_EXAMPLES), \
    'Each example must have instruction and response keys'
print('Schema check passed ✓')


Loaded 30 synthetic examples from /content/synth_data.json
First example instruction: What is a hash map and how do I use it in Python?...
Schema check passed ✓


### 3b. Public Dataset — CodeAlpaca-20k (Apache 2.0)

In [ ]:
from datasets import load_dataset

print("Loading CodeAlpaca-20k...")
code_alpaca = load_dataset('sahil2801/CodeAlpaca-20k', split='train')
print(f"Total: {len(code_alpaca)}")

DSA_KEYWORDS = [
    'python', 'array', 'list', 'dict', 'sort', 'search', 'tree', 'graph',
    'stack', 'queue', 'linked list', 'dynamic programming', 'recursion',
    'algorithm', 'function', 'class', 'string', 'sql', 'select',
    'hash', 'binary', 'complexity', 'fibonacci', 'palindrome', 'two sum',
    'heap', 'trie', 'bfs', 'dfs', 'dp', 'memoiz', 'sliding window',
    'pointer', 'binary search', 'merge sort', 'quick sort'
]

def is_relevant(ex):
    text = (ex.get('instruction','') + ' ' + ex.get('input','')).lower()
    output = ex.get('output', '')
    # FIX: raised min output length to 200 (was 150) to ensure pedagogical content
    # FIX: raised max to 2500 to allow fuller explanations
    return (any(kw in text for kw in DSA_KEYWORDS)
            and 200 < len(output) < 2500
            and ('def ' in output or 'explain' in text or 'how' in text or 'what' in text))

filtered = code_alpaca.filter(is_relevant)
print(f"After filtering: {len(filtered)}")

sample = filtered.shuffle(seed=42).select(range(min(280, len(filtered))))

def convert(ex):
    instr = ex['instruction']
    if ex.get('input', '').strip():
        instr += f"\n\nInput:\n{ex['input']}"
    return {'instruction': instr, 'response': ex['output']}

public_data = [convert(ex) for ex in sample]
print(f"Public data: {len(public_data)} examples")

Loading CodeAlpaca-20k...


README.md:   0%|          | 0.00/147 [00:00<?, ?B/s]

code_alpaca_20k.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/20022 [00:00<?, ? examples/s]

Total: 20022


Filter:   0%|          | 0/20022 [00:00<?, ? examples/s]

After filtering: 2647
Public data: 280 examples


In [ ]:
# Upsampled synthetic examples to dominate over CodeAlpaca
SYNTHETIC_UPSAMPLED = SYNTHETIC_EXAMPLES * 10  # 30 * 10 = 300 structured examples

# Filtered CodeAlpaca harder — only kept examples that already have
# numbered steps OR code blocks, so they're closer to our target format
def has_structure(ex):
    r = ex['response']
    has_code = '```' in r or 'def ' in r
    has_steps = any(f'{n}.' in r for n in range(1, 6))
    has_length = len(r.split()) > 100
    return (has_code or has_steps) and has_length

structured_public = [ex for ex in public_data if has_structure(ex)]
print(f'Structured public examples: {len(structured_public)}/{len(public_data)}')

ALL_DATA = SYNTHETIC_UPSAMPLED + structured_public
random.shuffle(ALL_DATA)
print(f'Total: {len(ALL_DATA)} ({len(SYNTHETIC_UPSAMPLED)} synthetic + {len(structured_public)} public)')
print(f'Synthetic ratio: {len(SYNTHETIC_UPSAMPLED)/len(ALL_DATA):.0%}')

Structured public examples: 14/280
Total: 314 (300 synthetic + 14 public)
Synthetic ratio: 96%


In [ ]:
import gc
import torch

# clear everything
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

# Verify clean state
print(f"VRAM Reserved: {torch.cuda.memory_reserved() / 1e9:.2f} GB")
print(f"VRAM Allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print("GPU State Reset. Please re-run the 'Model Loading' and 'Inference' cells.")

VRAM Reserved: 0.00 GB
VRAM Allocated: 0.00 GB
GPU State Reset. Please re-run the 'Model Loading' and 'Inference' cells.


## 4. Model Loading

4-bit NF4 QLoRA on T4 GPU.

In [ ]:
from huggingface_hub import login
login(token="Your_HF_TOKEN")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,   # saves ~0.4 bits/param extra
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    CFG['base_model'], token=True, trust_remote_code=True, padding_side='right')
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f"Vocab size: {tokenizer.vocab_size}")

# Separate left-padded tokenizer for batch inference
eval_tokenizer = AutoTokenizer.from_pretrained(
    CFG['base_model'], token=True, trust_remote_code=True, padding_side='left')
eval_tokenizer.pad_token = eval_tokenizer.eos_token

print("Loading model (4-bit QLoRA)...")
model = AutoModelForCausalLM.from_pretrained(
    CFG['base_model'],
    quantization_config=bnb_config,
    device_map='auto',
    token=True,
    trust_remote_code=True,
    attn_implementation='eager',       # flash_attention_2 not supported on T4
)
model.config.use_cache = False
model.config.pretraining_tp = 1
model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)


lora_config = LoraConfig(
    r=CFG['lora_r'],
    lora_alpha=CFG['lora_alpha'],
    target_modules=CFG['target_modules'],
    lora_dropout=CFG['lora_dropout'],
    bias='none',
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Expect ~0.5-1% of total params with r=16 across 7 modules

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

Loading tokenizer...


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/241M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

trainable params: 20,766,720 || all params: 2,635,108,608 || trainable%: 0.7881


### 3c. Format Dataset (requires tokenizer)


In [ ]:
def format_example(ex):
    messages = [
        {"role": "user",  "content": SYSTEM_PROMPT + "\n\n" + ex['instruction']},
        {"role": "model", "content": ex['response']},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    return {'text': text}

formatted = [format_example(ex) for ex in ALL_DATA]
print(f'Formatted {len(formatted)} examples')

# Length filter — 512 tokens for mixed code+text
before = len(formatted)
formatted = [ex for ex in formatted
             if 200 < len(ex['text']) < CFG['max_seq_length'] * 3]
print(f'Length filter removed {before - len(formatted)} examples')

# Safety filter
import re
UNSAFE = [r'\bhack\b', r'\bexploit\b', r'\bmalware\b']
formatted = [ex for ex in formatted
             if not any(re.search(p, ex['text'].lower()) for p in UNSAFE)]
print(f'After safety filter: {len(formatted)} examples')

from sklearn.model_selection import train_test_split
from datasets import Dataset
train_raw, test_raw = train_test_split(formatted, test_size=CFG['test_size'],
                                       random_state=42)
train_dataset = Dataset.from_list(train_raw)
test_dataset  = Dataset.from_list(test_raw)
print(f'Train: {len(train_dataset)}  |  Test: {len(test_dataset)}')

# Verify the template looks correct
sample = train_dataset[0]['text']
print(f'\nSample length: {len(sample)} chars')
print('First 200 chars:', repr(sample[:200]))
assert '<start_of_turn>model' in sample, 'Template missing from formatted text!'
print('Template structure check passed ✓')


Formatted 314 examples
Length filter removed 0 examples
After safety filter: 314 examples
Train: 282  |  Test: 32

Sample length: 2164 chars
First 200 chars: '<bos><start_of_turn>user\nYou are a programming tutor specializing in Python, Data Structures & Algorithms, and SQL.\n\nREQUIRED FORMAT — you MUST use this exact structure for every answer:\n**Goal** — on'
Template structure check passed ✓


## 5. Training

**Key fix:** Using `DataCollatorForCompletionOnlyLM` which masks prompt tokens from the loss — the model only learns to predict the *response* part, not the system prompt. This is essential for instruction tuning quality.

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer, DataCollatorForCompletionOnlyLM


# Tokenize a dummy full turn and find where '<start_of_turn>model' appears
_dummy = tokenizer.apply_chat_template(
    [{"role": "user", "content": "hi"}, {"role": "model", "content": "yo"}],
    tokenize=True, add_generation_prompt=False,
)
# Gemma-2 template: <bos>[107][106] user [108] ... [107][106] model [108] ... [107]
# We want the IDs for '<start_of_turn>model\n' as they appear in-context
_marker_str = '<start_of_turn>model\n'
_marker_ids = tokenizer.encode(_marker_str, add_special_tokens=False)
print(f'Response template token IDs: {_marker_ids}')
print(f'Decoded back: {tokenizer.decode(_marker_ids)!r}')

# Sanity check: verify these IDs actually appear in the dummy sequence
def _find_sublist(seq, sub):
    for i in range(len(seq) - len(sub) + 1):
        if seq[i:i+len(sub)] == sub: return i
    return -1

pos = _find_sublist(_dummy, _marker_ids)
assert pos != -1, (
    f'Template IDs {_marker_ids} NOT found in dummy sequence {_dummy}!\n'
    f'Try: _marker_ids = {_dummy[_dummy.index(106):_dummy.index(106)+3]}'
)
print(f'Template found at position {pos} in dummy sequence ✓')

collator = DataCollatorForCompletionOnlyLM(
    response_template=_marker_ids,
    tokenizer=tokenizer,
)

# Verify collator actually masks prompts
_sample_texts = [train_dataset[j]['text'] for j in range(min(2, len(train_dataset)))]
_sample_encoded = [
    tokenizer(t, truncation=True, max_length=CFG['max_seq_length'])
    for t in _sample_texts
]
_sample_batch = collator(_sample_encoded)
_labels = _sample_batch['labels'][0].tolist()
_non_masked = [l for l in _labels if l != -100]
assert len(_non_masked) > 0, (
    'FATAL: all labels are -100! Collator never found the response template.\n'
    'The model will receive zero gradient. Check template IDs and data format.'
)
print(f'Collator check: {len(_non_masked)} response tokens unmasked (training targets) ✓')
print(f'               {len([l for l in _labels if l == -100])} prompt tokens masked ✓')

training_args = TrainingArguments(
    output_dir=CFG['output_dir'],
    num_train_epochs=CFG['num_train_epochs'],
    per_device_train_batch_size=CFG['per_device_batch'],
    per_device_eval_batch_size=CFG['per_device_batch'],
    gradient_accumulation_steps=CFG['grad_accum_steps'],
    learning_rate=CFG['learning_rate'],
    lr_scheduler_type=CFG['lr_scheduler'],
    warmup_ratio=CFG['warmup_ratio'],
    weight_decay=CFG['weight_decay'],
    max_grad_norm=CFG['max_grad_norm'],
    bf16=CFG['bf16'],
    fp16=not CFG['bf16'],
    logging_steps=CFG['logging_steps'],
    save_steps=CFG['save_steps'],
    eval_steps=CFG['eval_steps'],
    evaluation_strategy='steps',
    save_strategy='steps',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    report_to='none',
    optim='paged_adamw_32bit',
    group_by_length=False,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
    args=training_args,
    data_collator=collator,
    dataset_text_field='text',
    max_seq_length=CFG['max_seq_length'],
    packing=False,
)

steps_per_epoch = len(train_dataset) // (CFG['per_device_batch'] * CFG['grad_accum_steps'])
print(f'Steps/epoch: {steps_per_epoch}')
print(f'Total steps: {steps_per_epoch * CFG["num_train_epochs"]}')


Response template token IDs: [106, 2516, 108]
Decoded back: '<start_of_turn>model\n'
Template found at position 7 in dummy sequence ✓
Collator check: 365 response tokens unmasked (training targets) ✓
               199 prompt tokens masked ✓


Map:   0%|          | 0/282 [00:00<?, ? examples/s]

Map:   0%|          | 0/32 [00:00<?, ? examples/s]

Steps/epoch: 35
Total steps: 175


In [ ]:
print("Starting training...")
result = trainer.train()
print(f"\nDone. Final train loss: {result.training_loss:.4f}")
trainer.log_metrics('train', result.metrics)
trainer.save_metrics('train', result.metrics)

Starting training...


Step,Training Loss,Validation Loss
100,0.032100,0.019386



Done. Final train loss: 0.2251
***** train metrics *****
  epoch                    =     4.9645
  total_flos               =  8295714GF
  train_loss               =     0.2251
  train_runtime            = 0:24:58.69
  train_samples_per_second =      0.941
  train_steps_per_second   =      0.117


In [ ]:
adapter_path = f"{CFG['output_dir']}/final_adapter"
trainer.model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f"Adapter saved to {adapter_path}")
for f in sorted(os.listdir(adapter_path)):
    print(f"  {f}: {os.path.getsize(adapter_path+'/'+f)/1e6:.2f} MB")

Adapter saved to ./prog_tutor_qlora/final_adapter
  README.md: 0.01 MB
  adapter_config.json: 0.00 MB
  adapter_model.safetensors: 83.12 MB
  special_tokens_map.json: 0.00 MB
  tokenizer.json: 17.53 MB
  tokenizer.model: 4.24 MB
  tokenizer_config.json: 0.05 MB


## 6. Inference / Demo

In [ ]:
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
tuned_model = trainer.model

# Check what device the model thinks it's on
print(f"model.device: {tuned_model.device}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA device count: {torch.cuda.device_count()}")

# Get the actual device from model parameters directly
actual_device = next(tuned_model.parameters()).device
print(f"Actual param device: {actual_device}")

@torch.inference_mode()
def ask_tutor(question, model, tokenizer, system=None,
              max_new_tokens=None, temperature=None):
    if system is None: system = SYSTEM_PROMPT
    if max_new_tokens is None: max_new_tokens = CFG['max_new_tokens']
    if temperature is None: temperature = CFG['temperature']

    messages = [{"role": "user", "content": system + "\n\n" + question}]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    # Use actual device from parameters instead of model.device
    device = next(model.parameters()).device
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    prompt_len = inputs['input_ids'].shape[1]

    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=CFG['top_p'],
        pad_token_id=tokenizer.eos_token_id,
        cache_implementation=None,
    )
    generated = out[0][prompt_len:]
    return tokenizer.decode(generated, skip_special_tokens=True)


tuned_model.eval()

print('Ready for inference')

model.device: cuda:0
CUDA available: True
CUDA device count: 1
Actual param device: cuda:0
Ready for inference


In [ ]:
DEMO_QS = [
    "What is a heap and when should I use min-heap vs max-heap?",
    "I'm getting a RecursionError in my code. How do I fix it?",
    "Explain the difference between greedy algorithms and dynamic programming.",
]

for i, q in enumerate(DEMO_QS, 1):
    print("\n" + "="*70)
    print(f"[Demo {i}] {q}")
    print("="*70)
    ans = ask_tutor(q, tuned_model, tokenizer)
    print(ans)


[Demo 1] What is a heap and when should I use min-heap vs max-heap?
**Goal** — Understand heaps and distinguish min-heap vs max-heap.

**Concept** — A heap is a tree-based data structure that always has the *minimum* (min-heap) or *maximum* (max-heap) value at the root. The key property: for every parent node, its children node are smaller (min-heap) or larger (max-heap). Only O(1) to extract the highest/lowest.

**Step-by-step**
| Operation | Min-Heap | Max-Heap |
|---|---|---|
| Insert | O(1) | O(1) |
| Extract Max/Min | O(1) | O(1) |
| Delete Max/Min | O(1) | O(1) |
| Copied | O(1) | O(1) |

**Worked Example**
```python
import heapq

# Min-heap to find the nearest polling place
queue = [-3, -2, -1, 0, 1, 2, 3]
print(heapq.nlargest(3, queue))   # [3, 2, 1]
print(heapq.nsmallest(3, queue))   # [-3, -2, -1]

# Max-heap for task scheduling (last-in first-out)
heapq.heappush(heap, 3)
print(heapq.heappop(heap))   # 3
```

**Checkpoint** — Why do we use a min-heap when finding the highest

## 7. Evaluation: Base vs Fine-tuned

In [ ]:
print("Loading base model for comparison...")
# use quantization here too to keep memory manageable
bnb_eval = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16)
base_model_eval = AutoModelForCausalLM.from_pretrained(
    CFG['base_model'], quantization_config=bnb_eval,
    device_map='auto', attn_implementation='eager')
base_model_eval.eval()
base_tok = AutoTokenizer.from_pretrained(CFG['base_model'])
if base_tok.pad_token is None:
    base_tok.pad_token = base_tok.eos_token
print("Base model loaded")

Loading base model for comparison...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Base model loaded


In [ ]:
EVAL_PROMPTS = [
    # Python internals & features —
    "What is the difference between `is` and `==` in Python?",
    "How does Python's garbage collector work?",
    "What are *args and **kwargs and when do you use them?",
    "What is the difference between @staticmethod and @classmethod?",
    "How does Python's `with` statement and context manager work?",
    "What is the GIL in Python and how does it affect multithreading?",
    "What is the difference between a shallow copy and a deep copy?",
    "How do Python's `__str__` and `__repr__` differ?",

    # DSA —
    "How does merge sort work and what is its time complexity?",
    "Explain the union-find (disjoint set) data structure.",
    "What is a min-heap and how do you implement one in Python?",
    "How does BFS differ from DFS and when should you use each?",
    "What is a circular queue and how do you implement it?",
    "Explain the two-pointer technique with an example.",
    "What is the difference between a BST and a balanced BST?",
    "How do you detect a cycle in a linked list?",
    "What is the knapsack problem and how do you solve it with DP?",
    "Explain counting sort and when it outperforms comparison sorts.",

    # SQL
    "What is the difference between INNER JOIN and LEFT JOIN?",
    "How do RANK() and DENSE_RANK() differ from ROW_NUMBER()?",
    "What is database normalization and what are the normal forms?",
    "How do you calculate a running total in SQL?",
    "What is the difference between WHERE and HAVING?",

    # Debugging —
    "I'm getting `RecursionError: maximum recursion depth exceeded`. How do I fix it?",
    "Why does `0.1 + 0.2 != 0.3` in Python?",
    "My function returns None unexpectedly. What are common causes?",

    # out-of-scope
    "Can you recommend a good diet plan for weight loss?",
    "Translate this sentence to French: I love programming.",
    "Who won the FIFA World Cup in 2022?",
    "Can you help me write a cover letter for a marketing job?",
    "What is the best investment strategy for 2025?",
]
print(f"Eval prompts: {len(EVAL_PROMPTS)} (spec requires >= 30) ✓")



Eval prompts: 31 (spec requires >= 30) ✓


In [ ]:
def generate_responses(prompts, model, tok, batch_size=8):
    all_responses = []
    for i in range(0, len(prompts), batch_size):
        batch_prompts = prompts[i:i+batch_size]
        formatted = [
            tok.apply_chat_template(
                [{"role": "user", "content": SYSTEM_PROMPT + "\n\n" + p}],
                tokenize=False, add_generation_prompt=True
            )
            for p in batch_prompts
        ]
        inputs = tok(
            formatted, return_tensors='pt', padding=True,
            truncation=True, max_length=768
        ).to(model.device)
        with torch.inference_mode():
            out = model.generate(
                **inputs,
                max_new_tokens=400,
                do_sample=False,
                pad_token_id=tok.eos_token_id,
                cache_implementation=None,   # disabled static cache
            )
        prompt_len = inputs['input_ids'].shape[1]
        for seq in out:
            gen = seq[prompt_len:]
            all_responses.append(tok.decode(gen, skip_special_tokens=True))
        print(f'  {min(i+batch_size, len(prompts))}/{len(prompts)}', end='\r')
    print()
    return all_responses

print('Generating base model responses...')
base_responses = generate_responses(EVAL_PROMPTS, base_model_eval, base_tok)

print('Generating fine-tuned responses...')
tuned_responses = generate_responses(EVAL_PROMPTS, tuned_model, eval_tokenizer)

print(f'Generated {len(base_responses)} base + {len(tuned_responses)} tuned responses')

Generating base model responses...
  31/31
Generating fine-tuned responses...
  31/31
Generated 31 base + 31 tuned responses


### 7a. Pedagogical Structure Score

In [ ]:
def ped_score(response):
    r = response.lower()
    s = {
        'has_goal'      : int('**goal' in r or 'goal —' in r or 'goal:' in r),
        'has_concept'   : int('**concept' in r or 'concept —' in r or 'concept:' in r),
        'step_by_step'  : int(any(p in r for p in ['step-by-step', 'step 1', '1.', 'first,', 'then,'])),
        'code_example'  : int('```' in response or 'def ' in response),
        'has_checkpoint': int('**checkpoint' in r or 'checkpoint —' in r or 'checkpoint:' in r),
        'explanation'   : int(len(response.split()) > 80),
    }
    s['total'] = sum(v for k, v in s.items() if k != 'total')
    return s

OOS_INDICES = list(range(len(EVAL_PROMPTS) - 5, len(EVAL_PROMPTS)))

base_scores  = [ped_score(r) for r in base_responses]
tuned_scores = [ped_score(r) for r in tuned_responses]

# Only score in-scope prompts (OOS responses shouldn't have the structure)
in_scope_idx = [i for i in range(len(EVAL_PROMPTS)) if i not in OOS_INDICES]
base_avg  = np.mean([base_scores[i]['total']  for i in in_scope_idx])
tuned_avg = np.mean([tuned_scores[i]['total'] for i in in_scope_idx])

print(f"Pedagogical Structure Score (0-6, in-scope prompts only):")
print(f"  Base model:       {base_avg:.2f}")
print(f"  Fine-tuned model: {tuned_avg:.2f}")
print(f"  Delta:            +{tuned_avg - base_avg:.2f}")

# Sub-metric breakdown
metrics = ['has_goal','has_concept','step_by_step','code_example','has_checkpoint']
print(f"\n{'Metric':<20} {'Base':>8} {'Tuned':>8} {'Delta':>8}")
print("-"*48)
for m in metrics:
    b = np.mean([base_scores[i][m]  for i in in_scope_idx])
    t = np.mean([tuned_scores[i][m] for i in in_scope_idx])
    print(f"{m:<20} {b:>8.2f} {t:>8.2f} {t-b:>+8.2f}")

Pedagogical Structure Score (0-6, in-scope prompts only):
  Base model:       5.69
  Fine-tuned model: 5.69
  Delta:            +0.00

Metric                   Base    Tuned    Delta
------------------------------------------------
has_goal                 1.00     1.00    +0.00
has_concept              1.00     1.00    +0.00
step_by_step             1.00     1.00    +0.00
code_example             1.00     1.00    +0.00
has_checkpoint           0.69     0.69    +0.00


### 7b. Out-of-Scope Refusal Quality

In [ ]:
REDIRECT_KW = [
    'outside', 'not within', 'specifically', 'programming tutor',
    'not in my scope', 'designed to help', 'redirect', "can't help",
    'i specialize', "i'm a programming", 'programming question',
    "that's outside", "outside my"
]

def good_refusal(r):
    return int(any(kw in r.lower() for kw in REDIRECT_KW))



base_ref  = [good_refusal(base_responses[i])  for i in OOS_INDICES]
tuned_ref = [good_refusal(tuned_responses[i]) for i in OOS_INDICES]

print(f"Out-of-scope Refusal ({len(OOS_INDICES)} prompts):")
print(f"  Base model:       {sum(base_ref)}/{len(OOS_INDICES)}")
print(f"  Fine-tuned model: {sum(tuned_ref)}/{len(OOS_INDICES)}")

print("\nPer-prompt:")
oos_prompts = [EVAL_PROMPTS[i] for i in OOS_INDICES]
for i, (p, br, tr) in enumerate(zip(oos_prompts, base_ref, tuned_ref)):
    mark = lambda v: '✓' if v else '✗'
    print(f"  [{mark(br)}/{mark(tr)}] {p[:60]}")

Out-of-scope Refusal (5 prompts):
  Base model:       0/5
  Fine-tuned model: 4/5

Per-prompt:
  [✗/✓] Can you recommend a good diet plan for weight loss?
  [✗/✗] Translate this sentence to French: I love programming.
  [✗/✓] Who won the FIFA World Cup in 2022?
  [✗/✓] Can you help me write a cover letter for a marketing job?
  [✗/✓] What is the best investment strategy for 2025?


### 7c. Code Unit Tests

In [ ]:
import json

with open("/content/unit_tests.json", "r") as f:
    CODE_TESTS = json.load(f)

def extract_and_run(response, test_code):
    # Try fenced python block first
    blocks = re.findall(r'```python(.*?)```', response, re.DOTALL)
    # Fall back to any fenced block
    if not blocks:
        blocks = re.findall(r'```(.*?)```', response, re.DOTALL)
    # Fall back to extracting def blocks directly from plain text
    if not blocks:
        lines = response.split('\n')
        in_block, block = False, []
        for line in lines:
            if re.match(r'^(def |class )', line):
                in_block = True
            if in_block:
                block.append(line)
                # stop at blank line after content
                if in_block and block and line == '' and len(block) > 2:
                    break
        if block:
            blocks = ['\n'.join(block)]

    if not blocks:
        return 'NO_CODE'

    # Try each extracted block until one passes
    for block in blocks:
        code_str = block + '\n' + test_code
        with tempfile.NamedTemporaryFile(suffix='.py', mode='w', delete=False) as f:
            f.write(code_str)
            fname = f.name
        try:
            r = subprocess.run([sys.executable, fname],
                               capture_output=True, text=True, timeout=10)
            if r.returncode == 0 and 'PASS' in r.stdout:
                return 'PASS'
        except subprocess.TimeoutExpired:
            return 'TIMEOUT'
        finally:
            os.unlink(fname)
    return f'FAIL({r.stderr[:60]})'

print(f"{'Question':<55} {'Base':<12} {'Tuned'}")
print('-' * 82)
bp = tp = 0
for tc in CODE_TESTS:
    br = ask_tutor(tc['q'], base_model_eval, base_tok)
    tr = ask_tutor(tc['q'], tuned_model, eval_tokenizer)
    bres = extract_and_run(br, tc['test'])
    tres = extract_and_run(tr, tc['test'])
    bp += int(bres == 'PASS')
    tp += int(tres == 'PASS')
    print(f"{tc['q'][:53]:<55} {bres:<12} {tres}")

print(f"\nPass rate  Base: {bp}/{len(CODE_TESTS)}  Fine-tuned: {tp}/{len(CODE_TESTS)}")

Question                                                Base         Tuned
----------------------------------------------------------------------------------
Write a Python function that returns a list of FizzBu   PASS         FAIL(  File "/tmp/tmplo7p666g.py", line 3
    ```
    ^
SyntaxErr)
Write a Python function to count the frequency of eac   PASS         PASS
Write a Python function to check if a string is a pal   PASS         PASS
Write a Python function to reverse the words in a sen   PASS         PASS
Write a Python function to check if a string has bala   PASS         PASS
Write a Python function to find the longest common pr   PASS         PASS
Write a Python function to perform binary search on a   PASS         PASS
Write a Python function to find two numbers in a list   PASS         PASS
Write a Python function to find the majority element    PASS         PASS
Write a Python function to rotate a list to the right   PASS         PASS

Pass rate  Base: 10/10  Fine-tuned: 9/1

### 7d. Summary Results Table

In [ ]:
print("\n" + "="*65)
print("EVALUATION RESULTS SUMMARY")
print("="*65)
print(f"{'Metric':<38} {'Base':>10} {'Fine-tuned':>12} {'Delta':>8}")
print("-"*68)
print(f"{'Ped. structure score (0-6)':<38} {base_avg:>10.2f} {tuned_avg:>12.2f} {tuned_avg-base_avg:>+8.2f}")
print(f"{'OOS refusal rate':<38} {sum(base_ref)/len(OOS_INDICES):>10.1%} {sum(tuned_ref)/len(OOS_INDICES):>12.1%}")
print(f"{'Code unit test pass rate':<38} {bp/len(CODE_TESTS):>10.1%} {tp/len(CODE_TESTS):>12.1%}")
print("="*68)


EVALUATION RESULTS SUMMARY
Metric                                       Base   Fine-tuned    Delta
--------------------------------------------------------------------
Ped. structure score (0-6)                   5.69         5.69    +0.00
OOS refusal rate                             0.0%        80.0%
Code unit test pass rate                   100.0%        90.0%


### 7e. Side-by-Side Examples (Base vs Fine-tuned)

In [ ]:
SBS = [
    "What is a hash map and how do you use it in Python?",
    "Explain binary search with code.",
    "I have a bug: while i < 10: print(i)  -- it runs forever. Help!",
    "What is the difference between a list and a tuple in Python?",
    "Can you help me write a love letter?",  # OOS redirect test
]

for i, prompt in enumerate(SBS, 1):
    b = ask_tutor(prompt, base_model_eval, base_tok)
    t = ask_tutor(prompt, tuned_model, eval_tokenizer)
    print(f"\n{'='*72}")
    print(f"[{i}] {prompt}")
    print("\n--- BASE MODEL ---")
    print(b)
    print("\n--- FINE-TUNED ---")
    print(t)


[1] What is a hash map and how do you use it in Python?

--- BASE MODEL ---
**Goal** — Learn about hash maps, also known as dictionaries, in Python.

**Concept** — Imagine a real-world dictionary where you can quickly look up a word by its definition. A hash map in Python is like that, but for data. It allows you to store data as key-value pairs and access values directly using their keys. 

**Step-by-step**
1. Dictionaries are like a real-world dictionary, where you have a word as a key and its definition as the value.
2. Each key must be unique, so we can identify a specific item in the map.
3. To store data, we use the `dict` constructor.
4. To access data, we use the key to find the corresponding value.

**Worked Example**
```python
my_map = {'apple': 1, 'banana': 2, 'cherry': 3}
print(my_map['apple'])  # Output: 1
```

**Checkpoint** — Can you explain how you would add a new key-value pair to the dictionary? 


--- FINE-TUNED ---
**Goal** — Understand what a hash map is and how t

## 8. Export

In [ ]:
README = '''
# prog-tutor-gemma2-2b-qlora
Fine-tuned Gemma-2-2b-it for Programming / DSA / SQL tutoring.

## Setup
```
pip install transformers peft bitsandbytes accelerate
```

## Load
```python
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import torch

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                         bnb_4bit_compute_dtype=torch.float16)
base = AutoModelForCausalLM.from_pretrained(
    'google/gemma-2-2b-it', quantization_config=bnb, device_map='auto')
model = PeftModel.from_pretrained(base, './adapter')
tok   = AutoTokenizer.from_pretrained('./adapter')
model.eval()
```
'''
with open(f'{adapter_path}/README.md', 'w') as f: f.write(README)
print("README written")

README written


In [ ]:
import shutil
zip_path = shutil.make_archive('prog_tutor_adapter', 'zip', adapter_path)
print(f"Zipped: {zip_path}  ({os.path.getsize(zip_path)/1e6:.1f} MB)")


Zipped: /content/prog_tutor_adapter.zip  (83.7 MB)
